In [10]:
%pip install aiofiles tqdm

DEPRECATION: amazon-textract-pipeline-pagedimensions 0.0.8 has a non-standard dependency specifier Pillow>=9.4.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of amazon-textract-pipeline-pagedimensions or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: amazon-textract-pipeline-pagedimensions 0.0.8 has a non-standard dependency specifier pypdf>=2.5.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of amazon-textract-pipeline-pagedimensions or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart th

In [1]:
import pandas as pd
import numpy as np
import os
import asyncio
import aiofiles
from tqdm.asyncio import tqdm_asyncio
import dotenv

dotenv.load_dotenv(dotenv.find_dotenv())

df = pd.read_csv("file_listing.csv")

In [2]:
df['category'].unique()

array(['Annual Reports', 'Quarterly Financial Statements',
       'General Meetings (AGM’s, Extra-Ordinary or Other Shareholders Meetings)',
       'Audited Financial Statements', 'Corporate Governance Policy',
       'General,APO / IPO', 'Articles,APO / IPO', 'Articles',
       'Mergers,Acquisitions', 'Acquisitions,Mergers',
       'Acquisitions,Articles', 'Audited Financial Statements,Articles',
       'General Meetings (AGM’s, Extra-Ordinary or Other Shareholders Meetings),Articles',
       'Articles,General Meetings (AGM’s, Extra-Ordinary or Other Shareholders Meetings)',
       'Annual Reports,Articles', 'Mergers,Acquisitions,Articles',
       'Mergers,Articles',
       'General Meetings (AGM’s, Extra-Ordinary or Other Shareholders Meetings),Acquisitions,Articles',
       'General,Junior Market Prospectus', 'Articles,General,APO / IPO',
       'Articles,Dividend Considerations', 'Articles,agm',
       'Annual Reports,General Meetings (AGM’s, Extra-Ordinary or Other Shareholders Me

## Master List of Available PDFs

In [3]:
import boto3
import pandas as pd
import os

def list_pdfs_from_prefix(s3_client, bucket_name, prefix):
    """List all PDFs in a given S3 bucket prefix"""
    pdfs = []
    paginator = s3_client.get_paginator('list_objects_v2')
    
    try:
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            if 'Contents' in page:
                for obj in page['Contents']:
                    if obj['Key'].lower().endswith('.pdf'):
                        pdfs.append({
                            'Bucket': bucket_name,
                            'Year': prefix.split('/')[-2],
                            'Key': obj['Key'],
                            'LastModified': obj['LastModified'],
                            'Size': obj['Size']
                        })
    except Exception as e:
        print(f"Error accessing prefix {prefix} in bucket {bucket_name}: {str(e)}")
    
    return pdfs

# Initialize S3 client
s3_client = boto3.client(
    's3',
    region_name='us-east-1',
    aws_access_key_id=os.getenv('JSE_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('JSE_SECRET_ACCESS_KEY')
)

# Bucket and prefixes
bucket_name = 'data-sphere-ssg'
prefixes = [
    'all-files/2022/',
    'all-files/2023/',
    'all-files/2024/'
]

# Get PDFs from all prefixes
all_pdfs = []
for prefix in prefixes:
    prefix_pdfs = list_pdfs_from_prefix(s3_client, bucket_name, prefix)
    all_pdfs.extend(prefix_pdfs)

# Convert to DataFrame
pdf_df = pd.DataFrame(all_pdfs)

# Optional: Sort by LastModified date
pdf_df = pdf_df.sort_values('LastModified', ascending=False)

# Save to CSV if needed
pdf_df.to_csv('s3_pdf_listing.csv', index=False)

## Annual Reports

In [27]:
# Get annual reports
annual_reports = df[df['category'].str.lower().str.contains("annual reports")]
# Clean company name
annual_reports["InstrumentName"] = annual_reports["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
annual_reports["InstrumentCode"] = annual_reports["InstrumentCode"].str.lower().str.replace(" ", "_")
# Extract year from post_name
annual_reports['extracted_year'] = annual_reports['post_name'].str.extract(r'(\d{4})')
# Convert post_date string to datetime (assuming format like "YYYY-MM-DD HH:MM:SS")
annual_reports['post_date'] = pd.to_datetime(annual_reports['post_date'])
# Fill missing years with post_date year - 1
annual_reports['extracted_year'] = annual_reports['extracted_year'].fillna(
    annual_reports['post_date'].dt.year - 1
).astype(str)
# Create new post_name
annual_reports["new_post_name"] = (annual_reports["InstrumentName"] + "-" + 
                                 annual_reports["InstrumentCode"] + "-annual_reports-" + 
                                 annual_reports["extracted_year"] + "-" + 
                                 annual_reports["post_date"].dt.strftime('%Y-%m-%d_%H-%M-%S') + 
                                 ".pdf")
# Save new names
new_names = annual_reports[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_annual_reports.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/1172997941.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annual_reports["InstrumentName"] = annual_reports["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/1172997941.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annual_reports["InstrumentCode"] = annual_reports["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipyke

## Prospectus

In [24]:
# Get prospectus
prospectus = df[df['category'].str.lower().str.contains("prospectus")]
# Clean company name
prospectus["InstrumentName"] = prospectus["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
prospectus["InstrumentCode"] = prospectus["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
prospectus['post_date'] = pd.to_datetime(prospectus['post_date'])
# Create new post_name
prospectus["new_post_name"] = (prospectus["InstrumentName"] + "-" + 
                                 prospectus["InstrumentCode"] + 
                                 "-prospectus-" + 
                                 prospectus["post_date"].dt.strftime('%Y-%m-%d') + 
                                 ".pdf")
# Save new names
new_names = prospectus[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_prospectus.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/2653795395.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prospectus["InstrumentName"] = prospectus["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/2653795395.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prospectus["InstrumentCode"] = prospectus["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/265379

## APO

In [21]:
# Get APO
apo = df[df['category'].str.lower().str.contains("apo")]
# Clean company name
apo["InstrumentName"] = apo["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
apo["InstrumentCode"] = apo["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
apo['post_date'] = pd.to_datetime(apo['post_date'])
# Create new post_name
apo["new_post_name"] = (apo["InstrumentName"] + "-" + 
                                 apo["InstrumentCode"] + 
                                 "-apo-" + 
                                 apo["post_date"].dt.strftime('%Y-%m-%d') + 
                                 ".pdf")
# Save new names
new_names = apo[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_apo.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/3958515630.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  apo["InstrumentName"] = apo["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/3958515630.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  apo["InstrumentCode"] = apo["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/3958515630.py:8: SettingWithCopyWa

## IPO

In [23]:
# Get IPO
ipo = df[df['category'].str.lower().str.contains("ipo")]
# Clean company name
ipo["InstrumentName"] = ipo["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
ipo["InstrumentCode"] = ipo["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
ipo['post_date'] = pd.to_datetime(ipo['post_date'])
# Create new post_name
ipo["new_post_name"] = (ipo["InstrumentName"] + "-" + 
                                 ipo["InstrumentCode"] + 
                                 "-ipo-" + 
                                 ipo["post_date"].dt.strftime('%Y-%m-%d') + 
                                 ".pdf")
# Save new names
new_names = ipo[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_ipo.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/4133240092.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipo["InstrumentName"] = ipo["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/4133240092.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipo["InstrumentCode"] = ipo["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/4133240092.py:8: SettingWithCopyWa

## Merger

In [25]:
# Get merger
merger = df[df['category'].str.lower().str.contains("merger")]
# Clean company name
merger["InstrumentName"] = merger["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
merger["InstrumentCode"] = merger["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
merger['post_date'] = pd.to_datetime(merger['post_date'])
# Create new post_name
merger["new_post_name"] = (merger["InstrumentName"] + "-" + 
                                 merger["InstrumentCode"] + 
                                 "-merger-" + 
                                 merger["post_date"].dt.strftime('%Y-%m-%d') + 
                                 ".pdf")
# Save new names
new_names = merger[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_merger.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/1678627198.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merger["InstrumentName"] = merger["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/1678627198.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merger["InstrumentCode"] = merger["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/1678627198.py:8: Setti

## Acquisitions

In [ ]:
# Get acquisitions
acquisitions = df[df['category'].str.lower().str.contains("acquisitions")]
# Clean company name
acquisitions["InstrumentName"] = acquisitions["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
acquisitions["InstrumentCode"] = acquisitions["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
acquisitions['post_date'] = pd.to_datetime(acquisitions['post_date'])
# Create new post_name
acquisitions["new_post_name"] = (acquisitions["InstrumentName"] + "-" + 
                                 acquisitions["InstrumentCode"] + 
                                 "-acquisitions-" + 
                                 acquisitions["post_date"].dt.strftime('%Y-%m-%d') + 
                                 ".pdf")
# Save new names
new_names = acquisitions[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_acquisitions.csv", index=False)


## Disposals

In [18]:
# Get disposals
disposals = df[df['category'].str.lower().str.contains("disposals")]
# Clean company name
disposals["InstrumentName"] = disposals["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
disposals["InstrumentCode"] = disposals["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
disposals['post_date'] = pd.to_datetime(disposals['post_date'])
# Create new post_name
disposals["new_post_name"] = (
    disposals["InstrumentName"] + "-" + 
    disposals["InstrumentCode"] + 
    "-disposals-" + 
    disposals["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
disposals["verified"] = "no"
# Save new names
new_names = disposals[["guid", "post_name", "post_date", "new_post_name", "verified"]]
new_names.to_csv("newly_named_disposals.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/702803398.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  disposals["InstrumentName"] = disposals["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/702803398.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  disposals["InstrumentCode"] = disposals["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/702803398

## Dividend Considerations

In [26]:
# Get dividend considerations
dividend_considerations = df[df['category'].str.lower().str.contains("dividend considerations")]
# Clean company name
dividend_considerations["InstrumentName"] = dividend_considerations["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
dividend_considerations["InstrumentCode"] = dividend_considerations["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
dividend_considerations['post_date'] = pd.to_datetime(dividend_considerations['post_date'])
# Create new post_name
dividend_considerations["new_post_name"] = (
    dividend_considerations["InstrumentName"] + "-" + 
    dividend_considerations["InstrumentCode"] + 
    "-dividend_considerations-" + 
    dividend_considerations["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = dividend_considerations[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_dividend_considerations.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/366440477.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dividend_considerations["InstrumentName"] = dividend_considerations["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_9153/366440477.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dividend_considerations["InstrumentCode"] = dividend_considerations["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r0

## Dividend Payments

In [19]:
# Get dividend payments
dividend_payments = df[df['category'].str.lower().str.contains("dividend payments")]
# Clean company name
dividend_payments["InstrumentName"] = dividend_payments["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
dividend_payments["InstrumentCode"] = dividend_payments["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
dividend_payments['post_date'] = pd.to_datetime(dividend_payments['post_date'])
# Create new post_name
dividend_payments["new_post_name"] = (
    dividend_payments["InstrumentName"] + "-" + 
    dividend_payments["InstrumentCode"] + 
    "-dividend_payments-" + 
    dividend_payments["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
dividend_payments["verified"] = "no"
# Save new names
new_names = dividend_payments[["guid", "post_name", "post_date", "new_post_name", "verified"]]
new_names.to_csv("newly_named_dividend_payments.csv", index=False)


## Disposals

In [ ]:
# Get disposals
disposals = df[df['category'].str.lower().str.contains("disposals")]
# Clean company name
disposals["InstrumentName"] = disposals["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
disposals["InstrumentCode"] = disposals["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
disposals['post_date'] = pd.to_datetime(disposals['post_date'])
# Create new post_name
disposals["new_post_name"] = (
    disposals["InstrumentName"] + "-" + 
    disposals["InstrumentCode"] + 
    "-disposals-" + 
    disposals["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = disposals[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_disposals.csv", index=False)


## Directors circular

In [3]:
# Get directors circular
directors_circular = df[df['category'].str.lower().str.contains("directors circular")]
# Clean company name
directors_circular["InstrumentName"] = directors_circular["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
directors_circular["InstrumentCode"] = directors_circular["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
directors_circular['post_date'] = pd.to_datetime(directors_circular['post_date'])
# Create new post_name
directors_circular["new_post_name"] = (
    directors_circular["InstrumentName"] + "-" + 
    directors_circular["InstrumentCode"] + 
    "-directors_circular-" + 
    directors_circular["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = directors_circular[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_directors_circular.csv", index=False)


## Bulletins

In [4]:
# Get bulletins
bulletins = df[df['category'].str.lower().str.contains("bulletins")]
# Clean company name
bulletins["InstrumentName"] = bulletins["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
bulletins["InstrumentCode"] = bulletins["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
bulletins['post_date'] = pd.to_datetime(bulletins['post_date'])
# Create new post_name
bulletins["new_post_name"] = (
    "jse_weekly_bulletin" + "-" + 
    bulletins["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = bulletins[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_bulletins.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3987180778.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bulletins["InstrumentName"] = bulletins["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3987180778.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bulletins["InstrumentCode"] = bulletins["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3987180

## Updates

In [5]:
# Get updates
updates = df[df['category'].str.lower().str.contains("updates")]
# Clean company name
updates["InstrumentName"] = updates["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
updates["InstrumentCode"] = updates["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
updates['post_date'] = pd.to_datetime(updates['post_date'])
# Create new post_name
updates["new_post_name"] = (
    "jse_updates" + "-" + 
    updates["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = updates[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_updates.csv", index=False)


## Acquisitions

In [6]:
# Get acquisitions
acquisitions = df[df['category'].str.lower().str.contains("acquisitions")]
# Clean company name
acquisitions["InstrumentName"] = acquisitions["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
acquisitions["InstrumentCode"] = acquisitions["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
acquisitions['post_date'] = pd.to_datetime(acquisitions['post_date'])
# Create new post_name
acquisitions["new_post_name"] = (
    acquisitions["InstrumentName"] + "-" + 
    acquisitions["InstrumentCode"] + 
    "-acquisitions-" + 
    acquisitions["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = acquisitions[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_acquisitions.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3045766933.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acquisitions["InstrumentName"] = acquisitions["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3045766933.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acquisitions["InstrumentCode"] = acquisitions["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_4

## Appointments

In [7]:
# Get appointments
appointments = df[df['category'].str.lower().str.contains("appointments")]
# Clean company name
appointments["InstrumentName"] = appointments["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
appointments["InstrumentCode"] = appointments["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
appointments['post_date'] = pd.to_datetime(appointments['post_date'])
# Create new post_name
appointments["new_post_name"] = (
    appointments["InstrumentName"] + "-" + 
    appointments["InstrumentCode"] + 
    "-appointments-" + 
    appointments["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = appointments[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_appointments.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3015394806.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  appointments["InstrumentName"] = appointments["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3015394806.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  appointments["InstrumentCode"] = appointments["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_4

## Change in Management

In [8]:
# Get change in management
change_in_management = df[df['category'].str.lower().str.contains("change in management")]
# Clean company name
change_in_management["InstrumentName"] = change_in_management["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
change_in_management["InstrumentCode"] = change_in_management["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
change_in_management['post_date'] = pd.to_datetime(change_in_management['post_date'])
# Create new post_name
change_in_management["new_post_name"] = (
    change_in_management["InstrumentName"] + "-" + 
    change_in_management["InstrumentCode"] + 
    "-change_in_management-" + 
    change_in_management["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = change_in_management[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_change_in_management.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/4240709154.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  change_in_management["InstrumentName"] = change_in_management["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/4240709154.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  change_in_management["InstrumentCode"] = change_in_management["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33

## General Meetings

In [9]:
# get general meetings or agm
general_meetings = df[df['category'].str.lower().str.contains("general meetings") | df['category'].str.lower().str.contains("agm")]
# Clean company name
general_meetings["InstrumentName"] = general_meetings["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
general_meetings["InstrumentCode"] = general_meetings["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
general_meetings['post_date'] = pd.to_datetime(general_meetings['post_date'])
# Create new post_name
general_meetings["new_post_name"] = (
    general_meetings["InstrumentName"] + "-" + 
    general_meetings["InstrumentCode"] + 
    "-general_meetings-" + 
    general_meetings["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)   
# Save new names
new_names = general_meetings[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_general_meetings.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/47491344.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  general_meetings["InstrumentName"] = general_meetings["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/47491344.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  general_meetings["InstrumentCode"] = general_meetings["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T

## NAV

In [10]:
# Get NAV
nav = df[df['category'].str.lower().str.contains("nav")]
# Clean company name
nav["InstrumentName"] = nav["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
nav["InstrumentCode"] = nav["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
nav['post_date'] = pd.to_datetime(nav['post_date'])
# Create new post_name
nav["new_post_name"] = (
    nav["InstrumentName"] + "-" + 
    nav["InstrumentCode"] + 
    "-nav-" + 
    nav["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)   
# Save new names
new_names = nav[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_nav.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/340728266.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nav["InstrumentName"] = nav["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/340728266.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nav["InstrumentCode"] = nav["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/340728266.py:8: SettingWithCopyWa

## Trading in Shares

In [11]:
# Get trading in shares
trading_in_shares = df[df['category'].str.lower().str.contains("trading in shares")]
# Clean company name
trading_in_shares["InstrumentName"] = trading_in_shares["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
trading_in_shares["InstrumentCode"] = trading_in_shares["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
trading_in_shares['post_date'] = pd.to_datetime(trading_in_shares['post_date'])
# Create new post_name
trading_in_shares["new_post_name"] = (
    trading_in_shares["InstrumentName"] + "-" + 
    trading_in_shares["InstrumentCode"] + 
    "-trading_in_shares-" + 
    trading_in_shares["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
# Save new names
new_names = trading_in_shares[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_trading_in_shares.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/811521532.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trading_in_shares["InstrumentName"] = trading_in_shares["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/811521532.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trading_in_shares["InstrumentCode"] = trading_in_shares["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm00

## Rights Issues circular

In [14]:
# Get rights issues circular
rights_issues_circular = df[df['category'].str.lower().str.contains("rights")]
# Clean company name
rights_issues_circular["InstrumentName"] = rights_issues_circular["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
rights_issues_circular["InstrumentCode"] = rights_issues_circular["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
rights_issues_circular['post_date'] = pd.to_datetime(rights_issues_circular['post_date'])
# Create new post_name
rights_issues_circular["new_post_name"] = (
    rights_issues_circular["InstrumentName"] + "-" + 
    rights_issues_circular["InstrumentCode"] + 
    "-rights_issues_circular-" + 
    rights_issues_circular["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)   
# Save new names
new_names = rights_issues_circular[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_rights_issues_circular.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/4159500781.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rights_issues_circular["InstrumentName"] = rights_issues_circular["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/4159500781.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rights_issues_circular["InstrumentCode"] = rights_issues_circular["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r0

## Take over big circular

In [13]:
# Get take over big circular
take_over_big_circular = df[df['category'].str.lower().str.contains("take over big circular")]
# Clean company name
take_over_big_circular["InstrumentName"] = take_over_big_circular["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
take_over_big_circular["InstrumentCode"] = take_over_big_circular["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
take_over_big_circular['post_date'] = pd.to_datetime(take_over_big_circular['post_date'])
# Create new post_name
take_over_big_circular["new_post_name"] = (
    take_over_big_circular["InstrumentName"] + "-" + 
    take_over_big_circular["InstrumentCode"] + 
    "-take_over_big_circular-" + 
    take_over_big_circular["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)   
# Save new names
new_names = take_over_big_circular[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_take_over_big_circular.csv", index=False)


## Regulatory Report

In [15]:
# Get regulatory report
regulatory_report = df[df['category'].str.lower().str.contains("regulatory")]
# Clean company name
regulatory_report["InstrumentName"] = regulatory_report["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
regulatory_report["InstrumentCode"] = regulatory_report["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
regulatory_report['post_date'] = pd.to_datetime(regulatory_report['post_date'])
# Create new post_name
regulatory_report["new_post_name"] = (
    regulatory_report["InstrumentName"] + "-" + 
    regulatory_report["InstrumentCode"] + 
    "-regulatory_report-" + 
    regulatory_report["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
regulatory_report["verified"] = "no"
# Save new names
new_names = regulatory_report[["guid", "post_name", "post_date", "new_post_name"]]
new_names.to_csv("newly_named_regulatory_report.csv", index=False)


## Other Company News

In [16]:
# Get other company news
other_company_news = df[df['category'].str.lower().str.contains("other company news")]
# Clean company name
other_company_news["InstrumentName"] = other_company_news["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
other_company_news["InstrumentCode"] = other_company_news["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
other_company_news['post_date'] = pd.to_datetime(other_company_news['post_date'])
# Create new post_name
other_company_news["new_post_name"] = (
    other_company_news["InstrumentName"] + "-" + 
    other_company_news["InstrumentCode"] + 
    "-other_company_news-" + 
    other_company_news["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
other_company_news["verified"] = "no"
# Save new names
new_names = other_company_news[["guid", "post_name", "post_date", "new_post_name", "verified"]]
new_names.to_csv("newly_named_other_company_news.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/1080975321.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  other_company_news["InstrumentName"] = other_company_news["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/1080975321.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  other_company_news["InstrumentCode"] = other_company_news["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxf

## Resignations and Retirements

In [17]:
# Get resignations and retirements
resignations_and_retirements = df[df['category'].str.lower().str.contains("resignations")]
# Clean company name
resignations_and_retirements["InstrumentName"] = resignations_and_retirements["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
resignations_and_retirements["InstrumentCode"] = resignations_and_retirements["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
resignations_and_retirements['post_date'] = pd.to_datetime(resignations_and_retirements['post_date'])
# Create new post_name
resignations_and_retirements["new_post_name"] = (
    resignations_and_retirements["InstrumentName"] + "-" + 
    resignations_and_retirements["InstrumentCode"] + 
    "-resignations_and_retirements-" + 
    resignations_and_retirements["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
resignations_and_retirements["verified"] = "no"
# Save new names
new_names = resignations_and_retirements[["guid", "post_name", "post_date", "new_post_name", "verified"]]
new_names.to_csv("newly_named_resignations_and_retirements.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3179228136.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  resignations_and_retirements["InstrumentName"] = resignations_and_retirements["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/3179228136.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  resignations_and_retirements["InstrumentCode"] = resignations_and_retirements["InstrumentCode"].str.lower().str.replace(" ", "

## Articles

In [24]:
# Get articles
articles = df[df['category'].str.lower().str.contains("articles")]
# Clean company name
articles["InstrumentName"] = articles["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
articles["InstrumentCode"] = articles["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
articles['post_date'] = pd.to_datetime(articles['post_date'])
# Create new post_name
articles["new_post_name"] = (
    articles["InstrumentName"] + "-" + 
    articles["InstrumentCode"] + 
    "-articles-" + 
    articles["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
articles["verified"] = "no" 
# Save new names
new_names = articles[["guid", "post_name", "post_date", "new_post_name", "verified"]]
new_names.to_csv("newly_named_articles.csv", index=False)


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/875396700.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  articles["InstrumentName"] = articles["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/875396700.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  articles["InstrumentCode"] = articles["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/875396700.py:

## Quarterly Financial Statements

In [35]:
# Get financial statements
financial_statements = df[df['category'].str.lower().str.contains("quarterly financial")]
# Clean company name
financial_statements["InstrumentName"] = financial_statements["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
financial_statements["InstrumentCode"] = financial_statements["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
financial_statements['post_date'] = pd.to_datetime(financial_statements['post_date'])
# Create new post_name
financial_statements["new_post_name"] = (
    financial_statements["InstrumentName"] + "-" + 
    financial_statements["InstrumentCode"] + 
    "-financial_statements-" + 
    financial_statements["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
financial_statements["verified"] = "no"
# Save new names
new_names = financial_statements[["guid", "post_name", "post_date", "new_post_name", "verified"]]
new_names.to_csv("newly_named_quarterly_financial_statements.csv", index=False)
print(f"Number of quarterly financial statements: {len(financial_statements)}")


Number of quarterly financial statements: 2461


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/1563552157.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  financial_statements["InstrumentName"] = financial_statements["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/1563552157.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  financial_statements["InstrumentCode"] = financial_statements["InstrumentCode"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33

## Audited Financial Statements

In [36]:
# Get audited financial statements
audited_financial_statements = df[df['category'].str.lower().str.contains("audited financial")]
# Clean company name
audited_financial_statements["InstrumentName"] = audited_financial_statements["InstrumentName"].str.lower().str.replace(" ", "_")
# Clean instrument code
audited_financial_statements["InstrumentCode"] = audited_financial_statements["InstrumentCode"].str.lower().str.replace(" ", "_")
# convert post_date to datetime
audited_financial_statements['post_date'] = pd.to_datetime(audited_financial_statements['post_date'])
# Create new post_name
audited_financial_statements["new_post_name"] = (
    audited_financial_statements["InstrumentName"] + "-" + 
    audited_financial_statements["InstrumentCode"] + 
    "-audited_financial_statements-" + 
    audited_financial_statements["post_date"].dt.strftime('%Y-%m-%d') + 
    ".pdf"
)
audited_financial_statements["verified"] = "no"
# Save new names
new_names = audited_financial_statements[["guid", "post_name", "post_date", "new_post_name", "verified"]]
new_names.to_csv("newly_named_audited_financial_statements.csv", index=False)
print(f"Number of audited financial statements: {len(audited_financial_statements)}")


Number of audited financial statements: 754


/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/1888333124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  audited_financial_statements["InstrumentName"] = audited_financial_statements["InstrumentName"].str.lower().str.replace(" ", "_")
/var/folders/rs/m7r05kdj3r33v6fmtrxfv5cm0000gn/T/ipykernel_49774/1888333124.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  audited_financial_statements["InstrumentCode"] = audited_financial_statements["InstrumentCode"].str.lower().str.replace(" ", "

## Progress

In [4]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os

load_dotenv()

def get_available_sheets():
    try:
        creds = Credentials(
            None,
            refresh_token=os.getenv("GOOGLE_REFRESH_TOKEN"),
            token_uri=os.getenv("GOOGLE_TOKEN_URI"),
            client_id=os.getenv("GOOGLE_CLIENT_ID"),
            client_secret=os.getenv("GOOGLE_CLIENT_SECRET")
        )
        service = build('sheets', 'v4', credentials=creds)
        
        # Get spreadsheet metadata
        sheet_metadata = service.spreadsheets().get(
            spreadsheetId=os.getenv("GOOGLE_SHEET_ID")
        ).execute()
        
        # Extract sheet names
        sheets = sheet_metadata.get('sheets', [])
        sheet_names = [sheet['properties']['title'] for sheet in sheets]
        
        return sheet_names
    except Exception as e:
        print(f"Error getting sheet names: {str(e)}")
        return []
    
available_sheets = get_available_sheets()
available_sheets


['newly_named_bulletins',
 'newly_named_updates',
 'newly_named_regulatory_report',
 'newly_named_acquisitions',
 'newly_named_apo',
 'newly_named_ipo',
 'newly_named_appointments',
 'newly_named_change_in_management',
 'newly_named_disposals',
 'newly_named_dividend_considerations',
 'newly_named_dividend_payments',
 'newly_named_general_meetings',
 'newly_named_merger',
 'newly_named_nav',
 'newly_named_other_company_news',
 'newly_named_resignations_and_retirements',
 'newly_named_trading_in_shares',
 'newly_named_annual_reports',
 'newly_named_directors_circular',
 'newly_named_prospectus',
 'newly_named_rights_issues_circular',
 'newly_named_take_over_big_circular',
 'newly_named_articles']

In [16]:
def get_sheet_data(sheet_name):
    """Get data from a specific sheet"""
    try:
        creds = Credentials(
            None,
            refresh_token=os.getenv("GOOGLE_REFRESH_TOKEN"),
            token_uri=os.getenv("GOOGLE_TOKEN_URI"),
            client_id=os.getenv("GOOGLE_CLIENT_ID"),
            client_secret=os.getenv("GOOGLE_CLIENT_SECRET")
        )
        service = build('sheets', 'v4', credentials=creds)
        
        # Get data from sheet
        result = service.spreadsheets().values().get(
            spreadsheetId=os.getenv("GOOGLE_SHEET_ID"),
            range=sheet_name
        ).execute()
        
        # Convert to DataFrame
        values = result.get('values', [])
        if not values:
            return pd.DataFrame()
            
        # Create DataFrame directly from all values
        df = pd.DataFrame(values)
        
        # Use first row as headers
        df.columns = df.iloc[0]
        df = df.iloc[1:]
        
        # Reset index
        df = df.reset_index(drop=True)
        
        return df
    
    except Exception as e:
        print(f"Error getting data from sheet {sheet_name}: {str(e)}")
        return pd.DataFrame()

# Get all sheet data and concatenate
all_data = []
for sheet_name in available_sheets:
    print(f"Processing sheet: {sheet_name}")
    sheet_df = get_sheet_data(sheet_name)
    if not sheet_df.empty:
        sheet_df['source_sheet'] = sheet_name  # Add source sheet name as column
        print(len(sheet_df[sheet_df["verified"] != "yes"]))
        all_data.append(sheet_df)

# Concatenate all dataframes
final_df = pd.concat(all_data, ignore_index=True)
total_rows = len(final_df)
print(f"Total rows: {total_rows}")
print(f"Percent of master sheet: {round(total_rows / len(pdf_df), 2)}")

# Drop rows where "verified" is not "yes"
final_df = final_df[final_df["verified"] == "yes"]
# drop duplicates
final_df = final_df.drop_duplicates(subset=["guid"])
verified_rows = len(final_df)

# Save to CSV
final_df.to_csv("combined_sheet_data.csv", index=False)


print(f"Number of verified: {verified_rows}")
print(f"Percent of verified: {round(verified_rows / len(pdf_df), 2)}")


Processing sheet: newly_named_bulletins
0
Processing sheet: newly_named_updates
Processing sheet: newly_named_regulatory_report
Processing sheet: newly_named_acquisitions
0
Processing sheet: newly_named_apo
8
Processing sheet: newly_named_ipo
5
Processing sheet: newly_named_appointments
44
Processing sheet: newly_named_change_in_management
23
Processing sheet: newly_named_disposals
0
Processing sheet: newly_named_dividend_considerations
17
Processing sheet: newly_named_dividend_payments
0
Processing sheet: newly_named_general_meetings
278
Processing sheet: newly_named_merger
15
Processing sheet: newly_named_nav
5
Processing sheet: newly_named_other_company_news
458
Processing sheet: newly_named_resignations_and_retirements
50
Processing sheet: newly_named_trading_in_shares
49
Processing sheet: newly_named_annual_reports
280
Processing sheet: newly_named_directors_circular
Processing sheet: newly_named_prospectus
0
Processing sheet: newly_named_rights_issues_circular
1
Processing sheet:

In [17]:
479+1+280+49+50+458+5+15+278+17+23+44+5+8

1712

## Miscellaneous

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain_google_vertexai import VertexAIEmbeddings

doc_type_loader = CSVLoader(file_path="Naming_Convention_Documents - Copy of Copy of Tagged Documents.csv")
docs = doc_type_loader.load()

embeddings = VertexAIEmbeddings(
    model_name="text-embedding-004",
)

doc_type_db = FAISS.from_documents(docs, embeddings)


In [4]:
from tqdm import tqdm

# At the start of your script
unique_categories = df["post_name"].unique()
category_mappings = {}

print("Pre-computing category mappings...")
for category in tqdm(unique_categories):
    clean_category = category.lower().replace(" ", "_")
    docs = doc_type_db.similarity_search(clean_category, k=1)
    doc_type = docs[0].page_content
    category_mappings[clean_category] = doc_type.split("\n")[1].replace("Document Type: ", "").strip()

Pre-computing category mappings...


100%|██████████| 4101/4101 [18:50<00:00,  3.63it/s]


In [5]:
async def process_row(row):
    file_loc = row["guid"]
    post_name = row["post_name"]
    
    # lower and clean instrument name
    instrument_name = row["InstrumentName"].lower().replace(" ", "_")
    instrument_code = row["InstrumentCode"].lower().replace(" ", "_")
    
    # clean category
    category = post_name.lower().replace(" ", "_")
    clean_doc_type = category_mappings[category]  # Simple dictionary lookup
    really_clean_doc_type = clean_doc_type.lower().replace(" ", "_")
    
    # remove time from post date
    post_date = row["post_date"].split(" ")[0]
    post_time = row['post_date'].split(" ")[1]
    post_date_year, post_date_month, post_date_day = post_date.split("-")
    
    # Create new location
    new_loc = f"all_files/{instrument_name}/{really_clean_doc_type}/{post_date_year}/{post_date_month}/"
    new_post_name = f"{instrument_name}_{instrument_code}_{really_clean_doc_type}_{post_date}.pdf"
    
    return file_loc, post_name, new_loc, new_post_name

async def main():
    tasks = []
    for _, row in df.iterrows():
        tasks.append(process_row(row))
    
    results = await tqdm_asyncio.gather(*tasks, desc="Processing files")
    old_guids, post_names, new_guids, new_post_names = zip(*results)
    
    new_df = pd.DataFrame({"old_guid": old_guids, "post_name": post_names, "new_guid": new_guids, "new_post_name": new_post_names})   
    new_df.to_csv("new_guids_2.csv", index=False)

# In Jupyter, you need to run this:
await main()

Processing files: 100%|██████████| 4889/4889 [00:00<00:00, 93580.22it/s]


# Approach 3

In [28]:
df = pd.read_csv("/Users/galbraithelroy/Documents/jse_pipeline/file_listing.csv")

In [29]:
df['post_date'] = pd.to_datetime(df['post_date'])

# get only posts in 2022
df_2022 = df[df['post_date'].dt.year == 2022]

In [30]:
len(df_2022)

1767

In [32]:
cats_2022 = df_2022['category'].str.lower().unique()
len(cats_2022)

129

In [37]:
df_2022 = df_2022[~df_2022['category'].str.lower().str.contains("quarterly financial")]
len(df_2022)


943